# 0.28 — `detect_theme`: run the funnel on any theme / date

Thin demo of the reusable wrapper in **`code/theme_detect.py`**. Feed a **corpus of headlines**
plus an **important date**, get back the tidy per-`(week, anchor)` dataframe with every funnel step:

`candidates → caught → promoted → llm_kept`  (+ `rank_candidate / rank_caught / rank_promoted / llm_judged`)

Every threshold is a keyword argument; the LLM gate is **off by default**.

**Corpus**: the single preprocessed file `output/news_corpus.parquet` (ADD-event dating, 2010–2025 —
built by `scripts/preprocess_news.py`; replay artifact proven in 0.25, layout in 0.26). The first cell
is a **gate**: it checks the corpus is built and up to date, and builds it from raw only if needed.
Never read `output/tmp/terms_{year}.parquet` in analyses — those are disposable build caches.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))                    # add code/ to the path
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
import pandas as pd
import polars as pl
from theme_detect import detect_theme
import preprocess_news as pp

CORPUS = pp.ensure()               # gate: builds missing years / re-merges ONLY if needed (no-op when certified)

def load_corpus(y0, y1):
    """[Headline, date, terms] slice of THE corpus, calendar years y0..y1 (lazy — never loads all years)."""
    return (pl.scan_parquet(CORPUS)
              .filter(pl.col("date").dt.year().is_between(y0, y1))
              .collect().to_pandas())

pd.set_option("display.max_colwidth", 46); pd.set_option("display.width", 200)
print("imported detect_theme from", detect_theme.__module__)

2010: certified add-event-v1 build exists — skipping
2011: certified add-event-v1 build exists — skipping
2012: certified add-event-v1 build exists — skipping
2013: certified add-event-v1 build exists — skipping
2014: certified add-event-v1 build exists — skipping
2015: certified add-event-v1 build exists — skipping
2016: certified add-event-v1 build exists — skipping
2017: certified add-event-v1 build exists — skipping
2018: certified add-event-v1 build exists — skipping
2019: certified add-event-v1 build exists — skipping
2020: certified add-event-v1 build exists — skipping
2021: certified add-event-v1 build exists — skipping
2022: certified add-event-v1 build exists — skipping
2023: certified add-event-v1 build exists — skipping
2024: certified add-event-v1 build exists — skipping
2025: certified add-event-v1 build exists — skipping
corpus up to date: news_corpus.parquet
imported detect_theme from theme_detect


## 1 · Quantum around the CHPX ETF date (2025-01-15)

Slice the corpus (any `[Headline, date, terms]` frame works; raw `[Headline, date]` is
extracted internally) and pass a quantum `theme_hint` to flag relevant anchors.

In [2]:
QUANTUM = r"\bionq\b|rigetti|\bd-wave\b|\bdwave\b|quantinuum|\bqubit|quantum comput|psiquantum|quantum advantage"
corpus = load_corpus(2022, 2025)

df = detect_theme(corpus, "2025-01-15", theme_hint=QUANTUM)      # LLM off by default
print(f"{len(df):,} rows · {int(df.caught.sum()):,} caught · {int(df.promoted.sum())} promoted · {int(df.is_theme.sum())} theme row(s)")
df[df.is_theme]

5,104 rows · 3,789 caught · 41 promoted · 1 theme row(s)


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
4875,2025-01-07/2025-01-13,ionq delta,"daily, daily volume, delta, hedge, ionq, o...",3,9,1.0,True,True,False,<NA>,IonQ Delta Hedge at 12% Daily Volume: Opti...,107,151,<NA>,False


In [3]:
df[df.promoted].sort_values("clustering")

,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
3670,2024-12-03/2024-12-09,hegseth,"trump, ernst, cbs, defense, fighting, hear...",9,34,0.343,False,True,True,<NA>,*HEGSETH SAYS HE HEARD FROM TRUMP TODAY TO...,5,4,4,True
3978,2024-12-10/2024-12-16,mangione,"luigi, luigi mangione, murder, suspect, su...",6,30,0.371,False,True,True,<NA>,*PROSECUTORS FILED MURDER & OTHER CHARGES ...,19,8,8,True
2374,2024-10-29/2024-11-04,altair engineering,"altair, engineering, bausch, lomb, survey,...",5,18,0.410,False,True,True,<NA>,"Altair Engineering, Bausch + Lomb Are Top ...",17,12,34,False
3437,2024-11-26/2024-12-02,apotea,"price, final, sek58, set, share, apotea fi...",7,25,0.429,False,True,True,<NA>,*APOTEA FINAL PRICE IN OFFERING IS SET TO ...,4,8,19,True
3155,2024-11-19/2024-11-25,groupe dynamite,"dynamite, groupe, canadian, fall, ahead, g...",9,23,0.429,False,True,True,<NA>,*GROUPE DYNAMITE IPO GOES AHEAD AT C$21/SH...,4,13,24,True
3145,2024-11-19/2024-11-25,lucid capital,"capital, lucid, investment, credit, atyr, ...",13,34,0.429,False,True,True,<NA>,Atyr Pharma Rated New Buy at Lucid Capital...,2,4,5,True
2454,2024-10-29/2024-11-04,fnz,"ahlsell, holding, parts, parts holding, ta...",3,11,0.436,False,True,True,<NA>,"Parts Holding, TAP, Ahlsell, FNZ",86,83,41,False
3190,2024-11-19/2024-11-25,mokingran,"hong, kong, hong kong, mokingran offers, o...",5,15,0.438,False,True,True,<NA>,*MOKINGRAN OFFERS ABOUT 44M SHARES IN HONG...,25,43,37,False
5004,2025-01-14/2025-01-20,loulo-gounkoto,"barrick, mali, barrick provides, loulo-gou...",3,16,0.457,False,True,True,<NA>,*BARRICK PROVIDES FURTHER UPDATE ON LOULO-...,42,14,36,False
1870,2024-10-15/2024-10-21,opella,"sanofi, talks, transfer, bpifrance, france...",19,84,0.467,False,True,True,<NA>,*PAI PARTNERS IS SAID TO SUBMIT REVISED BI...,1,1,2,True


## 2 · A different theme — genAI around the CHAT ETF (2023-05-18)

Swap the regex and the date; nothing else changes. Here we widen `detect_months` to **9** so the window
reaches back before ChatGPT's Nov-2022 birth (with the default 5-month window it is already in the
baseline, hence not *novel*) — a good illustration of a tunable knob.

In [4]:
GENAI = r"chatgpt|generative ai|\bgenai\b|large language model"
corpus_ai = load_corpus(2020, 2023)

ai = detect_theme(corpus_ai, "2023-05-18", theme_hint=GENAI, detect_months=9)
print(f"{len(ai):,} rows · {int(ai.caught.sum()):,} caught · {int(ai.promoted.sum())} promoted · {int(ai.is_theme.sum())} theme row(s)")
ai[ai.is_theme].head(20)                                          # chatgpt: caught -> promoted

9,895 rows · 7,433 caught · 68 promoted · 36 theme row(s)


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
4585,2023-01-03/2023-01-09,chatgpt,"microsoft, chatgpt requires, companies, po...",6,42,0.486,True,True,False,<NA>,ChatGPT Requires a Policy Response From Co...,6,1,<NA>,False
4792,2023-01-10/2023-01-16,chatgpt,"billion, investment, microsoft, billion in...",7,30,0.467,True,True,False,<NA>,Microsoft Weighs $10 Billion ChatGPT Inves...,6,2,<NA>,False
5022,2023-01-17/2023-01-23,chatgpt,"microsoft, chatgpt maker, maker, openai, i...",16,57,0.505,True,True,True,<NA>,*MICROSOFT TO ADD CHATGPT TO AZURE CLOUD S...,1,1,3,True
5098,2023-01-17/2023-01-23,chatgpt maker,"chatgpt, maker, microsoft, openai, investm...",7,12,0.773,True,True,False,<NA>,*MICROSOFT TO BOOST INVESTMENT IN CHATGPT ...,6,75,<NA>,False
5273,2023-01-24/2023-01-30,chatgpt,"microsoft, nvidia, altman, chatgpt creator...",10,57,0.524,True,True,False,<NA>,ChatGPT’s Microsoft Billions May Be Good f...,1,1,<NA>,False
5501,2023-01-31/2023-02-06,chatgpt,"month, openai, subscription, ai-related, c...",23,107,0.390,True,True,False,<NA>,"Samsung Expects New Memory Demand From AI,...",1,1,<NA>,False
5723,2023-01-31/2023-02-06,chatgpt pay,"chatgpt, introduces, month, openai, openai...",3,7,1.000,True,False,False,<NA>,*OPENAI INTRODUCES CHATGPT PAY SUBSCRIPTIO...,98,<NA>,<NA>,False
5790,2023-02-07/2023-02-13,chatgpt,"microsoft, chinese, answer, baidu, baidu s...",23,96,0.438,True,True,False,<NA>,Baidu Surges as Hope Mounts over Chinese A...,1,1,<NA>,False
5791,2023-02-07/2023-02-13,chatgpt-like,"bot, service, baidu, alibaba, developing, ...",15,44,0.457,True,True,False,<NA>,Baidu Surges After Affirming ChatGPT-Like ...,2,2,<NA>,False
5818,2023-02-07/2023-02-13,chatgpt-style,"baidu, baidu surges, bot, ernie, prepping,...",3,18,0.400,True,True,False,<NA>,Baidu Surges After Prepping ChatGPT-Style ...,102,27,<NA>,False


## 3 · Tuning knobs

Every threshold is a plain keyword argument — loosen the gates inline to catch smaller / faster signals.

In [5]:
loose = detect_theme(corpus, "2025-01-15", theme_hint=QUANTUM,
                     degree_min=6, cluster_max=0.7, persist_weeks=1)
print(f"looser gates: {len(loose):,} rows · {int(loose.caught.sum()):,} caught · {int(loose.promoted.sum())} promoted")
loose[loose.is_theme]

looser gates: 5,104 rows · 4,612 caught · 1206 promoted


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
4875,2025-01-07/2025-01-13,ionq delta,"daily, daily volume, delta, hedge, ionq, o...",3,9,1.0,True,True,False,<NA>,IonQ Delta Hedge at 12% Daily Volume: Opti...,107,151,<NA>,False


## 4 · Theme birth — crypto around the first bitcoin mania (2014-01-15)

The first bitcoin bubble: Silk Road shutdown **2013-10-02**, US Senate hearing **2013-11-18**,
bitcoin > $1,000 **2013-11-27**, China PBOC ban **2013-12-05**, Dogecoin born **2013-12-06**.
Headline flow: ~12/mo (Jan–Aug 2013) → 74 → 157 → 201/mo (Nov 2013–Jan 2014), a ~7× burst.
Investability markers *predate* the mania — Winklevoss ETF S-1 **2013-07-01**, GBTC launch
**2013-09-25** — the same product-led pattern as quantum.

Detection window (5 mo before `2014-01-15`) covers the whole wave; the 24-month baseline
(Aug 2011 – Aug 2013) now exists thanks to the 2010+ corpus and *contains the 2011 and Apr-2013
mini-bubbles* — so `bitcoin` itself is baseline vocabulary and can never be an anchor. The funnel
must find the wave through the vocabulary it minted: `litecoin`, `dogecoin`, new bigrams of old
words. That is the point of this test.

In [6]:
CRYPTO = r"bitcoin|litecoin|dogecoin|\bbtc\b|\bcrypto|virtual currenc|digital currenc|mt gox|\bmtgox\b|coinbase"
corpus_btc = load_corpus(2011, 2014)

btc = detect_theme(corpus_btc, "2014-01-15", theme_hint=CRYPTO)
print(f"{len(btc):,} rows · {int(btc.caught.sum()):,} caught · {int(btc.promoted.sum())} promoted · {int(btc.is_theme.sum())} theme row(s)")
btc[btc.is_theme].head(20)

12,853 rows · 10,472 caught · 100 promoted · 6 theme row(s)


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
7787,2013-11-12/2013-11-18,senate bitcoins,"agencies, agencies tell, benefits, bitcoin...",4,9,0.944,True,True,False,<NA>,U.S. Agencies Tell Senate Bitcoins Offer L...,129,436,<NA>,False
8931,2013-11-26/2013-12-02,bitcoin service,"bitcoin, fees, remittances, service, targe...",4,13,0.795,True,True,False,<NA>,Bitcoin Service Targets Kenyan Remittances...,109,171,<NA>,False
10131,2013-12-10/2013-12-16,bitcoins fail,"bitcoins, fail, nation, test, scandinavia,...",3,17,0.762,True,True,False,<NA>,Bitcoins Fail Real Money Test in Scandinav...,253,81,<NA>,False
10729,2013-12-17/2013-12-23,btc china,"btc, china, bitcoin, accept, bitcoin tradi...",5,25,0.648,True,True,False,<NA>,*BTC CHINA SAYS IT CAN'T ACCEPT NEW DEPOSI...,55,20,<NA>,False
11479,2013-12-24/2013-12-30,bitcoin users,"bitcoin, risks, users, bank, central, taiw...",4,15,0.714,True,True,False,<NA>,RBI Cautions Bitcoin Users About Risks of ...,40,54,<NA>,False
11695,2013-12-31/2014-01-06,bitcoin tops,"bitcoin, tops, zynga, currency, virtual, v...",3,16,0.648,True,True,False,<NA>,"Bitcoin Tops $1,000 Again on Adoption by Z...",82,41,<NA>,False


## 5 · The big mania — crypto around BLOK (2018-01-17)

BLOK (Amplify Blockchain ETF, the first crypto-theme ETF in the screener) launched **2018-01-17** —
the exact month of the biggest crypto news peak in the corpus. Burst: **11.2×** over the 2015–17
baseline (`crypto` 40×, `coinbase` 17×, `bitcoin` 12×). In-window: the ICO boom, the bitcoin-cash
fork (Aug 2017), CBOE/CME futures launches (**2017-12-10/17**), bitcoin ~$19.7k (Dec 17).

The baseline contains the 2015–16 lull *and* — via the 2010+ corpus — the earlier 2013–14 mania, so
`bitcoin` is old vocabulary. Expect the futures-, fork- and ICO-era mints instead.

In [7]:
CRYPTO18 = r"bitcoin|\bcrypto|blockchain|ethereum|litecoin|ripple|\bico\b|coin offering|coinbase"
corpus18 = load_corpus(2015, 2018)

mania = detect_theme(corpus18, "2018-01-17", theme_hint=CRYPTO18)
print(f"{len(mania):,} rows · {int(mania.caught.sum()):,} caught · {int(mania.promoted.sum())} promoted · {int(mania.is_theme.sum())} theme row(s)")
mania[mania.is_theme].head(20)

6,992 rows · 5,364 caught · 60 promoted · 71 theme row(s)


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
44,2017-08-22/2017-08-28,bitcoin winner,"bitcoin, crypto, million, wave, winner, ai...",3,14,0.780,True,True,False,<NA>,Bitcoin Winner Looking to Ride Crypto Wave...,65,45,<NA>,False
808,2017-09-05/2017-09-11,ico eu1b,"eu1b, ico, spgb",3,3,1.000,True,False,False,<NA>,ICO EU1b 4/2022 SPGB +8,108,<NA>,<NA>,False
1126,2017-09-12/2017-09-18,ico sek500m,"ico, sek500m, social, area",3,4,1.000,True,False,False,<NA>,ICO SEK500m 5Y Social Bond MS +53 Area,109,<NA>,<NA>,False
1311,2017-09-19/2017-09-25,hive blockchain,"blockchain, hive, financing, cannabis, can...",4,10,0.556,True,True,False,<NA>,"Cannabis Wheaton, Hive Blockchain, McEwen",53,160,<NA>,False
2105,2017-10-03/2017-10-09,harness blockchain,"blockchain, harness, tech, interbank, inte...",3,9,0.778,True,True,False,<NA>,"*MAS, ABS-LED GROUP TO HARNESS BLOCKCHAIN ...",115,213,<NA>,False
2784,2017-10-17/2017-10-23,bitcoin bank,"bank, bitcoin, florida, florida man, hacke...",3,10,1.000,True,True,False,<NA>,*FLORIDA MAN TO GET 16 MONTHS OVER BITCOIN...,123,185,<NA>,False
3220,2017-10-24/2017-10-30,bitcoin retreats,"appears, bitcoin, offshoot, retreats, cryp...",5,8,0.893,True,True,False,<NA>,Bitcoin Retreats as Another Offshoot of Cr...,30,250,<NA>,False
3388,2017-10-31/2017-11-06,bitcoin futures,"bitcoin, futures, floodgates, open, cme, c...",7,22,0.467,True,True,False,<NA>,*CME GROUP REPORTS LAUNCH OF BITCOIN FUTURES,12,26,<NA>,False
3568,2017-10-31/2017-11-06,riot blockchain,"blockchain, riot, bitcoin, bitcoin mining,...",4,10,0.733,True,True,False,<NA>,*RIOT BLOCKCHAIN IN PACT TO BUY 1200 BITCO...,54,163,<NA>,False
4652,2017-11-21/2017-11-27,fear bitcoin,"bitcoin, fad, fear, hedge-fund, hedge-fund...",3,11,0.927,True,True,False,<NA>,Remember Tamagotchi? Hedge-Fund Platforms ...,104,135,<NA>,False


## 6 · Re-emergence through new entities — crypto around BITQ (2021-05-12)

Old theme, new wave — the *discriminating* test. Total burst is a modest **5.6×**, but the wave's
new vocabulary is extreme: **`nft` 198×** (0.1 → 16.5/mo, Beeple's $69M Christie's sale Mar 2021),
**`coinbase` 30×** (IPO **2021-04-14**, in-window), dogecoin mania, Tesla's $1.5B bitcoin purchase
(Feb 2021). Meanwhile `bitcoin` / `crypto` / `blockchain` all sit in the 2019–20 baseline.

A naive volume detector fires on `bitcoin`; the strict-novelty funnel *cannot* — it must fire on the
NFT/Coinbase-era entities instead. If it does, novelty and volume are provably measuring different
things.

In [8]:
CRYPTO21 = r"\bnfts?\b|non-fungible|opensea|beeple|dogecoin|coinbase|bitcoin|\bcrypto|\bether"
corpus21 = load_corpus(2019, 2021)

re21 = detect_theme(corpus21, "2021-05-12", theme_hint=CRYPTO21)
print(f"{len(re21):,} rows · {int(re21.caught.sum()):,} caught · {int(re21.promoted.sum())} promoted · {int(re21.is_theme.sum())} theme row(s)")
re21[re21.is_theme].head(20)

7,919 rows · 5,786 caught · 60 promoted · 39 theme row(s)


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
405,2020-12-22/2020-12-28,cryptocurrency xrp,"cryptocurrency, sec, xrp, pending, reporte...",4,11,0.655,True,True,False,<NA>,Cryptocurrency XRP Slumps on Reported Pend...,23,63,<NA>,False
920,2021-01-05/2021-01-11,cryptocurrency-linked,"bitcoin, bitcoin wave, higher, ride, wave",3,5,1.000,True,False,False,<NA>,Cryptocurrency-Linked Stocks Ride the Bitc...,126,<NA>,<NA>,False
1421,2021-01-19/2021-01-25,bitcoin return,"bitcoin, doubt, flows, fund, return, slow,...",3,11,0.891,True,True,False,<NA>,"Bitcoin Return to $40,000 in Doubt as Gray...",132,126,<NA>,False
1546,2021-01-19/2021-01-25,understand bitcoin,"asset, bitcoin, faith-based, faith-based a...",3,8,1.000,True,True,False,<NA>,Want to Understand Bitcoin? It’s a Faith-B...,132,213,<NA>,False
2246,2021-02-02/2021-02-08,cryptocurrency accounts,"accounts, bank, bank orders, central, cryp...",3,12,0.864,True,True,False,<NA>,*NIGERIAN CENTRAL BANK ORDERS CLOSURE OF C...,130,106,<NA>,False
2376,2021-02-02/2021-02-08,dogecoin hits,"dogecoin, dogg, hits, musk, record, snoop,...",3,8,1.000,True,True,False,<NA>,Dogecoin Hits Another Record After Tweets ...,130,229,<NA>,False
2403,2021-02-02/2021-02-08,takes crypto,"bitcoin, bitcoin rally, crypto, rally, rec...",5,8,1.000,True,True,False,<NA>,Bitcoin Rally Takes Crypto Market Value to...,31,229,<NA>,False
2507,2021-02-09/2021-02-15,bitcoin bet,"bet, bitcoin, tesla, crypto, japan, korea,...",4,21,0.562,True,True,False,<NA>,"Crypto Stocks Rally in Japan, Korea After ...",58,9,<NA>,False
3173,2021-02-16/2021-02-22,bonds-for-bitcoin,"microstrategy, boosts, microstrategy boost...",3,7,0.762,True,False,False,<NA>,MicroStrategy Raises Bonds-for-Bitcoin Off...,131,<NA>,<NA>,False
3355,2021-02-23/2021-03-01,coinbase reveals,"coinbase, reveals, probes, public, seeks, ...",3,13,0.628,True,True,False,<NA>,Coinbase Reveals Profit in Direct Listing ...,167,90,<NA>,False


## 7 · Hard case, tiny volume — e-sports around GAMR (2016-03-09)

GAMR was the first video-game thematic ETF, launched into a real but *tiny* wave: total theme flow
only **1.4×** the baseline, but the word `esports` itself runs **7.7×** on a near-zero base
(0.6 → 4.8/mo). In-window: Activision buys Major League Gaming (**2016-01-04**), Turner/WME's
**ELEAGUE** announced, King acquisition closes (Feb 2016).

At ~5 relevant headlines/month the default gates (3 mentions/wk, degree ≥ 8, 2-week persistence) are
expected to be too strict — so this section runs **default and loosened gates side by side**: the
§3 knobs demonstrated on a genuinely marginal theme instead of a strong one.

In [9]:
ESPORTS = r"e-?sports?|eleague|major league gaming|\bmlg\b|competitive gaming|video ?gam"
corpus16 = load_corpus(2013, 2016)

es = detect_theme(corpus16, "2016-03-09", theme_hint=ESPORTS)
print(f"default gates: {len(es):,} rows · {int(es.caught.sum()):,} caught · {int(es.promoted.sum())} promoted · {int(es.is_theme.sum())} theme row(s)")

es_loose = detect_theme(corpus16, "2016-03-09", theme_hint=ESPORTS,
                        min_mentions=2, degree_min=5, persist_weeks=1, cluster_max=0.75)
print(f"loose gates:   {len(es_loose):,} rows · {int(es_loose.caught.sum()):,} caught · {int(es_loose.promoted.sum())} promoted · {int(es_loose.is_theme.sum())} theme row(s)")
es_loose[es_loose.is_theme].head(15)

default gates: 10,120 rows · 8,117 caught · 100 promoted · 1 theme row(s)
loose gates:   44,704 rows · 42,539 caught · 7338 promoted · 6 theme row(s)


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
773,2015-10-13/2015-10-19,russian e-sports,"e-sports, invest, russian, usmanov, fund, ...",2,10,0.800,True,True,False,<NA>,Usmanov Fund Plans to Invest ~$100m in Rus...,458,596,<NA>,False
2345,2015-10-20/2015-10-26,mlg co-founder,"co-founder, espn, mlg, bornstein, espn ste...",2,13,0.756,True,True,False,<NA>,*FORMER CEO OF ESPN STEVE BORNSTEIN & MLG ...,430,172,<NA>,False
3444,2015-10-20/2015-10-26,beleaguered authority,"authority, beleaguered, billion, billion t...",2,7,1.000,True,True,False,<NA>,Beleaguered Authority Favored for $20 Bill...,430,1293,<NA>,False
18472,2015-12-08/2015-12-14,creates e-sports,"arts, creates, division, e-sports, electro...",2,9,1.000,True,True,False,<NA>,Electronic Arts Creates E-Sports Division ...,504,900,<NA>,False
23585,2015-12-29/2016-01-04,e-sports event,"activision, e-sports, event, gaming, leagu...",2,9,1.000,True,True,False,<NA>,*ACTIVISION ACQUIRES E-SPORTS EVENT PIONEE...,207,383,<NA>,False
42664,2016-03-01/2016-03-07,yahoo esports,"esports, yahoo, launch, content, emphasis,...",3,9,0.750,True,True,True,<NA>,*YAHOO! ANNOUNCES LAUNCH OF YAHOO ESPORTS,181,840,6604,False


## 8 · Negative control — cyber around BUG (2019-10-25)

Chronic theme, no emergence: cyber news flow *declines* into the date (**0.7×** — 59/mo in the
detection window vs 80/mo in the baseline), the vocabulary has been baseline furniture since 2010,
and the wave minted no new entities. The funnel **should return zero theme anchors** — a detector
that fires here is detecting noise, not emergence. (The real cyber peaks — WannaCry 2017-05,
ransomware summer 2021 — are nowhere near any cyber-ETF launch.)

In [10]:
CYBER = r"cyber ?security|cyber-? ?attack|ransomware|data breach|malware|phishing|cybercrime"
corpus_cy = load_corpus(2017, 2019)

cy = detect_theme(corpus_cy, "2019-10-25", theme_hint=CYBER)
n_theme = int(cy.is_theme.sum())
print(f"{len(cy):,} rows · {int(cy.caught.sum()):,} caught · {int(cy.promoted.sum())} promoted · {n_theme} theme row(s)")
print("NEGATIVE CONTROL " + ("PASSED — funnel stays silent on a chronic, non-emerging theme"
                             if n_theme == 0 else f"FAILED — {n_theme} anchor(s) fired:"))
cy[cy.is_theme].head(10)

6,145 rows · 4,645 caught · 62 promoted · 1 theme row(s)
NEGATIVE CONTROL FAILED — 1 anchor(s) fired:


,week,anchor,partners,mentions,degree,clustering,is_theme,caught,promoted,llm_kept,example,rank_candidate,rank_caught,rank_promoted,llm_judged
1151,2019-06-25/2019-07-01,cybersecurity risk,"cybersecurity, medtronic, medtronic minime...",3,11,1.0,True,True,False,<NA>,FDA Warns on Medtronic MiniMed Insulin Pum...,115,97,<NA>,False
